# 03 -- Phase 2: Integration & Astrometry

Group calibrated frames, align them (`astroalign`), inverse-variance stack
them, and plate-solve the master with Astrometry.net. The ERR plane is warped
and combined alongside the data; the master gets a WCS.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG
# ============================================================
# One shared module rather than this cell copied into six notebooks, so a
# path is changed once and the notebooks cannot drift apart.
# Override any path with an environment variable; see workshop_config.py.
import importlib, os, sys

_here = os.path.dirname(os.path.abspath('workshop_config.py'))
if _here not in sys.path:
    sys.path.insert(0, _here)

# Reloaded, not merely imported. A kernel that imported workshop_config before
# the file was edited keeps serving the cached module, and the first name added
# since then fails much further down as a bare NameError -- which is exactly how
# `raw_frames()` broke for anyone whose kernel predated it.
import workshop_config
importlib.reload(workshop_config)
from workshop_config import *   # noqa: F403  (RAW_DIR, WORK_DIR, PHASE*_DIR, ...)

require_dataset()   # fails now, with the command that fixes it, not later
os.makedirs(WORK_DIR, exist_ok=True)
show_config()

## Run Phase 2
Reads `PHASE1_DIR` and writes the master stacks into `PHASE2_DIR`.

Plate solving needs sky data. If a full local index set is available (the
config cell points `CASSA_ASTROMETRY_INDEX` at one) nothing is downloaded;
otherwise the pipeline fetches just what this field needs -- a few MB for
ASTAP, ~165 MB of index files for Astrometry.net -- and caches it, so a second
run is offline either way.

In [ ]:
from cassa_photometry.config import load_config
from cassa_photometry.phase2_integration.pipeline import IntegrationPipeline
cfg = load_config()
pipe = IntegrationPipeline(PHASE1_DIR, output_dir=PHASE2_DIR, config=cfg)
pipe.setup()
pipe.execute()
run_dir = pipe.run_dir
print('Phase 2 output:', run_dir)

## Inspect the master stack + its WCS

In [ ]:
import glob, os, numpy as np
from astropy.wcs import WCS
from cassa_photometry.fits_utils import read_mef
masters = sorted(glob.glob(os.path.join(run_dir, 'Master_*.fits')))
sci, err, dq, hdr = read_mef(masters[0])
print('STACKCNT:', hdr.get('STACKCNT'), ' TOT_EXP:', hdr.get('TOT_EXP'))
w = WCS(hdr)
print('WCS celestial:', w.has_celestial)
ny, nx = sci.shape
print('Field centre:', w.pixel_to_world(nx/2, ny/2).to_string('hmsdms'))

### Exercise 1 -- how much deeper is the stack?

Compare the median `ERR` of the master with that of one calibrated frame in the
same filter. A stack of $N$ frames should be about $\sqrt{N}$ deeper.

| Find | Expected |
|---|---|
| depth gain, `ERR`(single) / `ERR`(master) | `TBD` |
| frames stacked, $N$ | `TBD` |

_Expected values come from the reference reduction; `TBD` until that run is fixed._

In [ ]:
from cassa_photometry.fits_utils import read_mef

# Fill in the blanks marked TODO. Everything else is scaffolding.
N = hdr.get('STACKCNT')
filt = hdr.get('FILTER')

# One calibrated frame in the same filter, so the comparison is like for like.
single_path = None
for f in sorted(glob.glob(os.path.join(PHASE1_DIR, 'calibrated_*.fits'))):
    if read_mef(f)[3].get('FILTER') == filt:
        single_path = f
        break
_, single_err, _, _ = read_mef(single_path)

med_single = float(np.nanmedian(single_err))
med_master = FILL_IN     # TODO 1: the same statistic for the master's ERR plane
depth_gain = FILL_IN     # TODO 2: how many times deeper the stack is

print(f"filter {filt}, N = {N} frames stacked\n")
print(f"median ERR, single frame : {med_single:.3f} e-")
print(f"median ERR, master       : {med_master:.3f} e-")
print(f"depth gain               : {depth_gain:.3f}   (sqrt(N) = {np.sqrt(N):.3f})")

### Exercise 2 -- what did the plate solve buy us?

Read the plate scale out of the WCS and work out the field of view.

| Find | Expected |
|---|---|
| plate scale from the WCS | `TBD` arcsec/pixel |
| field of view | `TBD` arcmin |

In [ ]:
from astropy.wcs.utils import proj_plane_pixel_scales

# Fill in the blanks marked TODO. Everything else is scaffolding.
# The WCS was measured against real stars, so it is the authority on the scale --
# not SECPIX, which is whatever the acquisition software was told to write.
scale = FILL_IN          # TODO 1: arcsec/pixel from the WCS (proj_plane_pixel_scales gives deg)
fov_x = FILL_IN          # TODO 2: field of view along x, in arcmin

print(f"plate scale, WCS    : {scale[0]:.4f} x {scale[1]:.4f} arcsec/pixel")
print(f"plate scale, SECPIX : {hdr.get('SECPIX')} arcsec/pixel  (what the camera claimed)")
print(f"image size          : {nx} x {ny} pixels")
print(f"field of view       : {fov_x:.2f} x {ny * scale[1] / 60.0:.2f} arcmin")